## Imports

In [ ]:
import numpy as np
import gc
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from src.utils.preprocessing import prepare_dataset, normalize_features

import sys
import os
import importlib
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Reload the preprocessing module to get the latest version
if 'src.utils.preprocessing' in sys.modules:
    importlib.reload(sys.modules['src.utils.preprocessing'])

## 1. Load and preprocess (VERY CONSERVATIVE MEMORY APPROACH)

In [ ]:

print("Loading data with minimal memory footprint...")
try:
    # Note: prepare_dataset now generates RUL labels automatically (countdown from n-1 to 0)
    result = prepare_dataset("../data/SMARTPDM_dataset.csv")
    
    if result is None or len(result) != 2:
        raise ValueError("prepare_dataset failed to return valid data")
    
    X_raw, y = result
    print(f"Raw dataset: X={X_raw.shape}, y={y.shape}")
    print(f"RUL range: {y.min()} to {y.max()}")
    
    # AGGRESSIVE MEMORY OPTIMIZATION: Use only 2% of data to prevent crashes
    dataset_size_gb = X_raw.nbytes / (1024**3)
    print(f"Dataset size: {dataset_size_gb:.2f} GB")
    
    # Use much smaller sample for stable training
    sample_ratio = 0.02  # Only 2% of data
    sample_size = max(1000, int(len(X_raw) * sample_ratio))  # At least 1000 samples
    print(f"Using conservative sample: {sample_ratio*100}% ({sample_size:,} samples)")
    
    # Random sampling
    np.random.seed(42)  # For reproducibility
    indices = np.random.choice(len(X_raw), sample_size, replace=False)
    X_raw = X_raw[indices]
    y = y[indices]
    
    print(f"Sampled dataset: X={X_raw.shape}, y={y.shape}")
    print(f"New dataset size: {X_raw.nbytes / (1024**3):.3f} GB")
    
except Exception as e:
    print(f"Error loading dataset: {e}")
    raise





## 2. Scale RUL target to [0, 1]

In [ ]:
max_rul = np.max(y)
min_rul = np.min(y)
print(f"RUL range: {min_rul} to {max_rul}")
y_scaled = y / max_rul
print(f"Scaled RUL range: {y_scaled.min():.3f} to {y_scaled.max():.3f}")

## 3. Normalize input features

In [ ]:
print("Normalizing features...")
X_norm, scaler = normalize_features(X_raw)
del X_raw  # cleanup immediately
gc.collect()

## 4. Train/val/test split

In [ ]:
print("Splitting dataset...")
X_train, X_temp, y_train, y_temp = train_test_split(
    X_norm, y_scaled, test_size=0.3, random_state=42, stratify=None  # Remove stratify for small samples
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)
del X_norm, X_temp, y_temp
gc.collect()

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Memory usage - Train: {X_train.nbytes/(1024**2):.1f}MB")


## 5. Define SMALLER model

In [ ]:
def build_lightweight_rul_model(input_shape):
    """Smaller model to reduce memory usage"""
    inputs = layers.Input(shape=input_shape)
    
    # Reduce model complexity
    x = layers.Conv1D(16, kernel_size=5, activation='relu')(inputs)  # Reduced filters
    x = layers.MaxPooling1D(pool_size=4)(x)  # Larger pooling
    x = layers.Conv1D(32, kernel_size=3, activation='relu')(x)  # Reduced filters
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation='relu')(x)  # Reduced neurons
    x = layers.Dropout(0.2)(x)  # Add dropout for regularization
    output = layers.Dense(1, name='rul')(x)
    
    model = models.Model(inputs=inputs, outputs=output)
    return model

print("Building lightweight model...")
model = build_lightweight_rul_model(X_train.shape[1:])
model.compile(optimizer='adam', loss='mae', metrics=['mae', 'mse'])
model.summary()

## 6. Train with conservative settings

In [ ]:
print("Training model with conservative settings...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=3,  # Very few epochs
    batch_size=16,  # Very small batch size
    verbose=1
)

## 7. Predict & unscale

In [ ]:
print("Making predictions...")
y_pred_scaled = model.predict(X_test, batch_size=32).flatten()
y_pred_real = y_pred_scaled * max_rul
y_test_real = y_test * max_rul

## 8. Evaluate

In [ ]:
mae = mean_absolute_error(y_test_real, y_pred_real)
mse = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)  # Calculate RMSE manually
print(f"\nTest MAE (real RUL): {mae:.2f}")
print(f"Test RMSE (real RUL): {rmse:.2f}")
print(f"Test MSE (real RUL): {mse:.2f}")

## 9. Plot

In [ ]:
plt.figure(figsize=(12,6))

# Plot 1: Predictions vs ground truth
plt.subplot(1,2,1)
n_samples = min(200, len(y_test_real))
plt.plot(y_test_real[:n_samples], label='True RUL', alpha=0.7)
plt.plot(y_pred_real[:n_samples], label='Predicted RUL', alpha=0.7)
plt.title("RUL Prediction vs Ground Truth")
plt.xlabel("Sample")
plt.ylabel("Remaining Useful Life")
plt.legend()
plt.grid(True)

# Plot 2: Scatter plot
plt.subplot(1,2,2)
plt.scatter(y_test_real, y_pred_real, alpha=0.5)
plt.plot([y_test_real.min(), y_test_real.max()], [y_test_real.min(), y_test_real.max()], 'r--', lw=2)
plt.xlabel('True RUL')
plt.ylabel('Predicted RUL')
plt.title('True vs Predicted RUL')
plt.grid(True)

plt.tight_layout()
plt.show()

print("Training completed successfully!")